# ЛР № 3. Профилирование и качество данных Seoul Bike Sharing Demand

Участок `raw → staging` сквозного проекта. Основной вопрос из [Project Brief](../docs/project_brief.md):
как различается средний почасовой спрос на прокат велосипедов по часам суток в разные сезоны и в рабочие,
выходные и праздничные дни?

Одна строка исходного файла - один час одной календарной даты: суммарное число аренд за этот час во всей
системе проката вместе с погодными и календарными признаками. Бизнес-ключ - пара `Date` × `Hour`;
его уникальность проверяет правило `date_hour_unique`.

Перед запуском поднимите стенд (`docker compose up -d`) и выберите ядро из `.venv` проекта.
Выполняйте ячейки сверху вниз. Для проверки повторяемости перезапустите ядро и снова выполните все ячейки.

<a id="input"></a>
## Шаг 1. Параметры, подключение и входные данные

Следующая ячейка - параметры запуска: входной raw-объект, аналитический период и выходной префикс.
`ingested_on` - логическая дата загрузки Airflow, а не период наблюдений. Аналитический период задан
по дате наблюдения и охватывает весь годовой ряд: с 2017-12-01 включительно до 2018-12-01 не включительно.
Справочники в этой работе не используются.

In [1]:
RAW_PATH = "s3://raw/seoul_bike_sharing/ingested_on=2026-10-24/SeoulBikeData.csv"
PERIOD_START = "2017-12-01"  # первая дата периода, включается
PERIOD_END = "2018-12-01"  # дата после конца периода, не включается
OUTPUT_PREFIX = "s3://staging/seoul_bike_sharing/source_ingested_on=2026-10-24"

Ячейка ниже находит корень проекта при запуске из корня или из каталога `notebooks`, загружает
настройки S3 из `.env` без вывода секретов и использует функции DuckDB из ЛР № 2.

Заголовок `SeoulBikeData.csv` записан в кодировке Latin-1 (символ `°`), а DuckDB 1.1 читает CSV только
в UTF-8. Поэтому имена столбцов берёт функция `csv_header` из первой строки того же S3-объекта,
а данные читаются со второй строки. Все значения читаются как строки (`all_varchar`): правила качества
проверяют исходный текст, а типы назначаются явно при подготовке staging.

In [2]:
import os
import sys
from datetime import date
from html import escape
from pathlib import Path

import boto3
import duckdb
from dotenv import load_dotenv
from IPython.display import HTML, display

PROJECT_ROOT = next(
    parent
    for parent in (Path.cwd(), *Path.cwd().parents)
    if (parent / "pyproject.toml").is_file() and (parent / "src/analysis/read_raw.py").is_file()
)
load_dotenv(PROJECT_ROOT / ".env", override=True)
required_settings = ("S3_ENDPOINT", "S3_ACCESS_KEY", "S3_SECRET_KEY")
missing_settings = [name for name in required_settings if not os.environ.get(name)]
if missing_settings:
    raise ValueError("Не заданы настройки: " + ", ".join(missing_settings))

sys.path.insert(0, str(PROJECT_ROOT / "src"))
from analysis.read_raw import configure_s3, csv_header  # noqa: E402

SOURCE_DATE = RAW_PATH.split("/ingested_on=", 1)[1].split("/", 1)[0]
date.fromisoformat(SOURCE_DATE)
if OUTPUT_PREFIX != f"s3://staging/seoul_bike_sharing/source_ingested_on={SOURCE_DATE}":
    raise ValueError("Выходной префикс не соответствует логической дате raw-объекта.")


def literal(value):
    return "'" + str(value).replace("'", "''") + "'"


def identifier(name):
    return '"' + name.replace('"', '""') + '"'


def show_table(headers, rows):
    header = "".join(f"<th>{escape(str(item))}</th>" for item in headers)
    body = "".join(
        "<tr>" + "".join(f"<td>{escape(str(item))}</td>" for item in row) + "</tr>"
        for row in rows
    )
    display(HTML(f"<table><thead><tr>{header}</tr></thead><tbody>{body}</tbody></table>"))


endpoint = os.environ["S3_ENDPOINT"]
client = boto3.client(
    "s3",
    endpoint_url=endpoint if endpoint.startswith("http") else f"http://{endpoint}",
    aws_access_key_id=os.environ["S3_ACCESS_KEY"],
    aws_secret_access_key=os.environ["S3_SECRET_KEY"],
    region_name="us-east-1",
)
raw_bucket, raw_key = RAW_PATH.removeprefix("s3://").split("/", 1)
raw_before = client.head_object(Bucket=raw_bucket, Key=raw_key)

connection = duckdb.connect()
configure_s3(connection)
raw_columns = csv_header(connection, RAW_PATH)
expected_columns = [
    "Date", "Rented Bike Count", "Hour", "Temperature(°C)", "Humidity(%)", "Wind speed (m/s)",
    "Visibility (10m)", "Dew point temperature(°C)", "Solar Radiation (MJ/m2)", "Rainfall(mm)",
    "Snowfall (cm)", "Seasons", "Holiday", "Functioning Day",
]
if raw_columns != expected_columns:
    raise ValueError(f"Неожиданный заголовок raw-файла: {raw_columns}")

names_sql = "[" + ", ".join(literal(name) for name in raw_columns) + "]"
connection.execute(f"""
    CREATE OR REPLACE TEMP VIEW raw_data AS
    SELECT * FROM read_csv(
        {literal(RAW_PATH)}, header = false, skip = 1, names = {names_sql},
        all_varchar = true, hive_partitioning = false
    )
""")
show_table(
    ["Вход", "Логическая дата", "Начало периода", "Конец периода (не включается)", "Выход"],
    [(RAW_PATH, SOURCE_DATE, PERIOD_START, PERIOD_END, OUTPUT_PREFIX)],
)
show_table(raw_columns, connection.execute("SELECT * FROM raw_data LIMIT 3").fetchall())

Вход,Логическая дата,Начало периода,Конец периода (не включается),Выход
s3://raw/seoul_bike_sharing/ingested_on=2026-10-24/SeoulBikeData.csv,2026-10-24,2017-12-01,2018-12-01,s3://staging/seoul_bike_sharing/source_ingested_on=2026-10-24


Date,Rented Bike Count,Hour,Temperature(°C),Humidity(%),Wind speed (m/s),Visibility (10m),Dew point temperature(°C),Solar Radiation (MJ/m2),Rainfall(mm),Snowfall (cm),Seasons,Holiday,Functioning Day
01/12/2017,254,0,-5.2,37,2.2,2000,-17.6,0,0,0,Winter,No Holiday,Yes
01/12/2017,204,1,-5.5,38,0.8,2000,-17.6,0,0,0,Winter,No Holiday,Yes
01/12/2017,173,2,-6,39,1,2000,-17.7,0,0,0,Winter,No Holiday,Yes


<a id="profile"></a>
## Шаг 2. Профиль исходных данных

Для профиля выбраны поля, нужные для метрик и срезов основного вопроса: составной ключ `Date` × `Hour`,
мера `Rented Bike Count`, признак работы проката `Functioning Day`, срезы `Seasons` и `Holiday`.
Пятое поле - `Humidity(%)`: по нему в Project Brief выдвинута гипотеза о сбоях метеонаблюдений,
важных для погодных срезов уточняющих вопросов.

Профиль строится по исходным строкам до преобразований и исключения записей. В raw-представлении все
значения - строки; в таблице показан также тип, который DuckDB определяет автоматически.
Пропуском считается `NULL` или пустая строка.

In [3]:
raw_count = connection.execute("SELECT count(*) FROM raw_data").fetchone()[0]
detected_types = {
    row[0]: row[1]
    for row in connection.execute(f"""
        DESCRIBE SELECT * FROM read_csv(
            {literal(RAW_PATH)}, header = false, skip = 1, names = {names_sql},
            hive_partitioning = false
        )
    """).fetchall()
}
show_table(["Строк", "Полей"], [(raw_count, len(raw_columns))])

profile_fields = (
    "Date", "Hour", "Rented Bike Count", "Functioning Day", "Seasons", "Holiday", "Humidity(%)",
)
profile_rows = []
for field in profile_fields:
    column = identifier(field)
    missing_count = connection.execute(
        f"SELECT count(*) FROM raw_data WHERE NULLIF(trim({column}), '') IS NULL"
    ).fetchone()[0]
    examples = connection.execute(
        f"SELECT DISTINCT {column} FROM raw_data WHERE {column} IS NOT NULL ORDER BY 1 LIMIT 2"
    ).fetchall()
    profile_rows.append((
        field, detected_types[field], ", ".join(row[0] for row in examples),
        raw_count, missing_count, f"{missing_count / raw_count:.2%}",
    ))
show_table(
    ["Поле", "Тип по автоопределению DuckDB", "Примеры", "Проверено", "Пропусков", "Доля"],
    profile_rows,
)

Строк,Полей
8760,14


Поле,Тип по автоопределению DuckDB,Примеры,Проверено,Пропусков,Доля
Date,DATE,"01/01/2018, 01/02/2018",8760,0,0.00%
Hour,BIGINT,"0, 1",8760,0,0.00%
Rented Bike Count,BIGINT,"0, 10",8760,0,0.00%
Functioning Day,BOOLEAN,"No, Yes",8760,0,0.00%
Seasons,VARCHAR,"Autumn, Spring",8760,0,0.00%
Holiday,VARCHAR,"Holiday, No Holiday",8760,0,0.00%
Humidity(%),BIGINT,"0, 10",8760,0,0.00%


In [4]:
coverage = connection.execute("""
    SELECT
        min(try_strptime("Date", '%d/%m/%Y')::DATE),
        max(try_strptime("Date", '%d/%m/%Y')::DATE),
        count(DISTINCT "Date"),
        count(*) - count(DISTINCT ("Date", "Hour")) AS key_repeats,
        count(*) - (SELECT count(*) FROM (SELECT DISTINCT * FROM raw_data)) AS exact_repeats
    FROM raw_data
""").fetchone()
hours_per_date = connection.execute("""
    SELECT min(rows_per_date), max(rows_per_date)
    FROM (SELECT "Date", count(*) AS rows_per_date FROM raw_data GROUP BY "Date")
""").fetchone()
show_table(
    ["Первая дата", "Последняя дата", "Дат", "Повторов Date × Hour", "Полных повторов строк",
     "Мин. часов на дату", "Макс. часов на дату"],
    [(*coverage, *hours_per_date)],
)

season_distribution = connection.execute("""
    SELECT
        "Seasons",
        count(*) AS hours,
        count(*) FILTER (WHERE "Functioning Day" = 'Yes') AS functioning_hours,
        count(*) FILTER (WHERE "Functioning Day" = 'No') AS non_functioning_hours,
        count(DISTINCT "Date") FILTER (WHERE "Functioning Day" = 'No') AS dates_with_stops
    FROM raw_data
    GROUP BY "Seasons"
    ORDER BY min(try_strptime("Date", '%d/%m/%Y'))
""").fetchall()
show_table(
    ["Сезон", "Часов", "Прокат работал", "Прокат не работал", "Дат с остановкой"],
    season_distribution,
)

stops = connection.execute("""
    SELECT
        try_strptime("Date", '%d/%m/%Y')::DATE AS observed_on,
        count(*) AS non_functioning_hours,
        min(TRY_CAST("Hour" AS INTEGER)) AS first_hour,
        max(TRY_CAST("Hour" AS INTEGER)) AS last_hour,
        sum(TRY_CAST("Rented Bike Count" AS INTEGER)) AS rentals
    FROM raw_data
    WHERE "Functioning Day" = 'No'
    GROUP BY 1
    ORDER BY 1
""").fetchall()
show_table(["Дата остановки", "Часов без проката", "Первый час", "Последний час", "Аренд"], stops)

Первая дата,Последняя дата,Дат,Повторов Date × Hour,Полных повторов строк,Мин. часов на дату,Макс. часов на дату
2017-12-01,2018-11-30,365,0,0,24,24


Сезон,Часов,Прокат работал,Прокат не работал,Дат с остановкой
Winter,2160,2160,0,0
Spring,2208,2160,48,2
Summer,2208,2208,0,0
Autumn,2184,1937,247,11


Дата остановки,Часов без проката,Первый час,Последний час,Аренд
2018-04-11,24,0,23,0
2018-05-10,24,0,23,0
2018-09-18,24,0,23,0
2018-09-19,24,0,23,0
2018-09-28,24,0,23,0
2018-09-30,24,0,23,0
2018-10-02,24,0,23,0
2018-10-04,24,0,23,0
2018-10-06,7,0,6,0
2018-10-09,24,0,23,0


### Наблюдения для отчёта

- Прокат не работал 295 часов на 13 датах, и во всех этих часах `Rented Bike Count = 0`. 247 таких часов
  приходятся на осень (11,3 % осенних часов), а 2018-10-06 остановлены только часы 0-6. Если считать
  нерабочие часы обычными, средний спрос осенью окажется заниженным, поэтому по решению Project Brief они не
  участвуют в метриках спроса. Согласованность признака и числа аренд проверяет правило
  `non_functioning_hour_has_zero_rentals`.
- Документация UCI расходится с файлом: дата записана как `dd/mm/yyyy`, признак работы - `Yes`/`No`.
  Автоопределение DuckDB само распознаёт дату и превращает `Yes`/`No` в `BOOLEAN`, но для staging правила
  разбора заданы явно, чтобы результат не зависел от автоопределения.

Повторы по бизнес-ключу считаются как число строк минус число различных пар `Date` × `Hour`,
полные повторы - как число строк минус число различных строк по всем 14 полям.

<a id="rules"></a>
## Шаг 3. Правила качества

Девять правил ниже проверяют поля профиля. Для каждого указаны идентификатор, поля, проверяемые строки,
условие, порог, критичность, действие и связь с задачей. Единица порога у всех правил - число нарушивших
строк; допустимый максимум задан переменной в ячейке правила и участвует в сравнении. Превышение
фиксируется, когда число нарушений больше порога.

Порог оценивает набор в целом, а действие применяется к каждой нарушившей строке независимо от превышения.
Доля считается от числа проверенных строк. Правила охватывают полноту (правило 1), допустимость значений
(2-5 и 9), уникальность ключа (6) и согласованность связанных полей (7 и 8).

Вспомогательное представление `parsed_raw` разбирает исходные строки, не изменяя данные: дату по формату
`%d/%m/%Y`, час, число аренд и влажность. Целое значение принимается, только если строка состоит из цифр:
`TRY_CAST('7.5' AS INTEGER)` в DuckDB округляет значение до 8, поэтому одного приведения типа недостаточно.
`key_rows` - число строк с той же парой «дата × час».

In [5]:
connection.execute("""
    CREATE OR REPLACE TEMP VIEW parsed_raw AS
    WITH parsed AS (
        SELECT *,
            try_strptime(trim("Date"), '%d/%m/%Y')::DATE AS parsed_date,
            CASE WHEN regexp_full_match(trim("Hour"), '[0-9]+')
                 THEN TRY_CAST(trim("Hour") AS INTEGER) END AS parsed_hour,
            CASE WHEN regexp_full_match(trim("Rented Bike Count"), '[0-9]+')
                 THEN TRY_CAST(trim("Rented Bike Count") AS INTEGER) END AS parsed_count,
            TRY_CAST(trim("Humidity(%)") AS DOUBLE) AS parsed_humidity
        FROM raw_data
    )
    SELECT *, count(*) OVER (PARTITION BY parsed_date, parsed_hour) AS key_rows
    FROM parsed
""")

RULE_HEADERS = [
    "Правило", "Проверено", "Нарушений", "Доля", "Порог", "Сравнение", "Критичность", "Действие",
]
rules = {}


def record_rule(rule_id, population, violation, checked, violations, maximum, severity, action,
                reason):
    share = f"{violations / checked:.2%}" if checked else "N/A"
    comparison = ("Превышен" if violations > maximum else "Не превышен") if checked else "N/A"
    row = (rule_id, checked, violations, share, f"count ≤ {maximum}", comparison, severity, action)
    rules[rule_id] = {
        "population": population, "violation": violation, "action": action, "reason": reason,
        "row": row,
    }
    show_table(RULE_HEADERS, [row])

### Правило 1. `required_fields_present`

- Поля: `Date`, `Hour`, `Rented Bike Count`, `Functioning Day`, `Seasons`, `Holiday`.
- Проверяемые строки: все строки raw.
- Условие: все шесть полей заполнены (не `NULL` и не пустая строка).
- Порог: число нарушений не больше `max_missing_required = 0`.
- Критичность: `critical`. Действие: `quarantine`.
- Связь с задачей: без даты и часа нельзя определить час суток, сезон и тип дня; без числа аренд нет меры
  спроса; без признака работы нельзя решить, участвует ли час в среднем спросе.

In [6]:
max_missing_required = 0
required_population = "true"
required_violation = """
    NULLIF(trim("Date"), '') IS NULL OR NULLIF(trim("Hour"), '') IS NULL
    OR NULLIF(trim("Rented Bike Count"), '') IS NULL OR NULLIF(trim("Functioning Day"), '') IS NULL
    OR NULLIF(trim("Seasons"), '') IS NULL OR NULLIF(trim("Holiday"), '') IS NULL
"""
checked, violations = connection.execute(f"""
    SELECT count(*), count(*) FILTER (WHERE {required_violation})
    FROM parsed_raw
    WHERE {required_population}
""").fetchone()
record_rule("required_fields_present", required_population, required_violation, checked,
            violations, max_missing_required, "critical", "quarantine",
            "Не заполнено обязательное поле")

Правило,Проверено,Нарушений,Доля,Порог,Сравнение,Критичность,Действие
required_fields_present,8760,0,0.00%,count ≤ 0,Не превышен,critical,quarantine


### Правило 2. `date_valid_in_period`

- Поле: `Date`.
- Проверяемые строки: строки с заполненной датой.
- Условие: строка разбирается по формату `%d/%m/%Y` в существующую дату, и эта дата лежит в аналитическом
  периоде `PERIOD_START ≤ дата < PERIOD_END`.
- Порог: число нарушений не больше `max_bad_date = 0`.
- Критичность: `critical`. Действие: `quarantine`.
- Связь с задачей: по дате определяются сезон, день недели и тип дня; дата вне годового ряда нарушит
  сравнение сезонов.

In [7]:
max_bad_date = 0
date_population = """NULLIF(trim("Date"), '') IS NOT NULL"""
date_violation = f"""
    parsed_date IS NULL
    OR parsed_date < DATE {literal(PERIOD_START)} OR parsed_date >= DATE {literal(PERIOD_END)}
"""
checked, violations = connection.execute(f"""
    SELECT count(*), count(*) FILTER (WHERE {date_violation})
    FROM parsed_raw
    WHERE {date_population}
""").fetchone()
record_rule("date_valid_in_period", date_population, date_violation, checked, violations,
            max_bad_date, "critical", "quarantine", "Дата не разбирается или вне периода")
day_after_month = connection.execute("""
    SELECT count(*) FROM raw_data WHERE TRY_CAST(split_part("Date", '/', 1) AS INTEGER) > 12
""").fetchone()[0]
show_table(["Строк, где первое число даты больше 12"], [(day_after_month,)])

Правило,Проверено,Нарушений,Доля,Порог,Сравнение,Критичность,Действие
date_valid_in_period,8760,0,0.00%,count ≤ 0,Не превышен,critical,quarantine


"Строк, где первое число даты больше 12"
5304


### Правило 3. `hour_in_range`

- Поле: `Hour`.
- Проверяемые строки: строки с заполненным часом.
- Условие: строка состоит из цифр, а значение лежит от 0 до 23.
- Порог: число нарушений не больше `max_bad_hour = 0`.
- Критичность: `critical`. Действие: `quarantine`.
- Связь с задачей: час суток - основной срез суточного профиля спроса.

In [8]:
max_bad_hour = 0
hour_population = """NULLIF(trim("Hour"), '') IS NOT NULL"""
hour_violation = "parsed_hour IS NULL OR parsed_hour > 23"
checked, violations = connection.execute(f"""
    SELECT count(*), count(*) FILTER (WHERE {hour_violation})
    FROM parsed_raw
    WHERE {hour_population}
""").fetchone()
record_rule("hour_in_range", hour_population, hour_violation, checked, violations, max_bad_hour,
            "critical", "quarantine", "Час не является целым числом от 0 до 23")

Правило,Проверено,Нарушений,Доля,Порог,Сравнение,Критичность,Действие
hour_in_range,8760,0,0.00%,count ≤ 0,Не превышен,critical,quarantine


### Правило 4. `rented_count_non_negative_integer`

- Поле: `Rented Bike Count`.
- Проверяемые строки: строки с заполненным числом аренд.
- Условие: строка состоит из цифр, то есть значение - целое неотрицательное число в диапазоне `INTEGER`.
- Порог: число нарушений не больше `max_bad_count = 0`.
- Критичность: `critical`. Действие: `quarantine`.
- Связь с задачей: число аренд - мера среднего и медианного почасового спроса.

In [9]:
max_bad_count = 0
count_population = """NULLIF(trim("Rented Bike Count"), '') IS NOT NULL"""
count_violation = "parsed_count IS NULL"
checked, violations = connection.execute(f"""
    SELECT count(*), count(*) FILTER (WHERE {count_violation})
    FROM parsed_raw
    WHERE {count_population}
""").fetchone()
record_rule("rented_count_non_negative_integer", count_population, count_violation, checked,
            violations, max_bad_count, "critical", "quarantine",
            "Число аренд не является целым неотрицательным числом")

Правило,Проверено,Нарушений,Доля,Порог,Сравнение,Критичность,Действие
rented_count_non_negative_integer,8760,0,0.00%,count ≤ 0,Не превышен,critical,quarantine


### Правило 5. `categories_valid`

- Поля: `Seasons`, `Holiday`, `Functioning Day`.
- Проверяемые строки: строки, где заполнены все три поля.
- Условие: значения точно совпадают со справочником категорий источника: `Seasons` - `Winter`, `Spring`,
  `Summer`, `Autumn`; `Holiday` - `Holiday`, `No Holiday`; `Functioning Day` - `Yes`, `No`.
- Порог: число нарушений не больше `max_bad_category = 0`.
- Критичность: `critical`. Действие: `quarantine`.
- Связь с задачей: сезон и тип дня - срезы основного вопроса, а признак работы определяет, какие часы
  участвуют в метриках. Неизвестное значение нельзя однозначно отнести ни к одной группе.

In [10]:
max_bad_category = 0
category_population = """
    NULLIF(trim("Seasons"), '') IS NOT NULL AND NULLIF(trim("Holiday"), '') IS NOT NULL
    AND NULLIF(trim("Functioning Day"), '') IS NOT NULL
"""
category_violation = """
    "Seasons" NOT IN ('Winter', 'Spring', 'Summer', 'Autumn')
    OR "Holiday" NOT IN ('Holiday', 'No Holiday')
    OR "Functioning Day" NOT IN ('Yes', 'No')
"""
checked, violations = connection.execute(f"""
    SELECT count(*), count(*) FILTER (WHERE {category_violation})
    FROM parsed_raw
    WHERE {category_population}
""").fetchone()
record_rule("categories_valid", category_population, category_violation, checked, violations,
            max_bad_category, "critical", "quarantine", "Значение категории вне справочника")
show_table(["Поле", "Значение", "Строк"], connection.execute("""
    SELECT 'Seasons', "Seasons", count(*) FROM raw_data GROUP BY 2
    UNION ALL SELECT 'Holiday', "Holiday", count(*) FROM raw_data GROUP BY 2
    UNION ALL SELECT 'Functioning Day', "Functioning Day", count(*) FROM raw_data GROUP BY 2
    ORDER BY 1, 2
""").fetchall())

Правило,Проверено,Нарушений,Доля,Порог,Сравнение,Критичность,Действие
categories_valid,8760,0,0.00%,count ≤ 0,Не превышен,critical,quarantine


Поле,Значение,Строк
Functioning Day,No,295
Functioning Day,Yes,8465
Holiday,Holiday,432
Holiday,No Holiday,8328
Seasons,Autumn,2184
Seasons,Spring,2208
Seasons,Summer,2208
Seasons,Winter,2160


### Правило 6. `date_hour_unique`

- Поля: составной бизнес-ключ `Date` × `Hour`.
- Проверяемые строки: строки с корректными датой и часом (правила 2 и 3 по формату).
- Условие: каждая пара «дата × час» встречается один раз. Нарушением считаются все строки пары, которая
  встречается несколько раз: без бизнес-правила нельзя выбрать «правильную» из них.
- Порог: число нарушений не больше `max_key_repeats = 0`.
- Критичность: `critical`. Действие: `quarantine`.
- Связь с задачей: повтор часа завысит число аренд и исказит средний спрос в своей группе.

In [11]:
max_key_repeats = 0
key_population = "parsed_date IS NOT NULL AND parsed_hour BETWEEN 0 AND 23"
key_violation = "key_rows > 1"
checked, violations = connection.execute(f"""
    SELECT count(*), count(*) FILTER (WHERE {key_violation})
    FROM parsed_raw
    WHERE {key_population}
""").fetchone()
record_rule("date_hour_unique", key_population, key_violation, checked, violations,
            max_key_repeats, "critical", "quarantine", "Пара дата × час повторяется")

Правило,Проверено,Нарушений,Доля,Порог,Сравнение,Критичность,Действие
date_hour_unique,8760,0,0.00%,count ≤ 0,Не превышен,critical,quarantine


### Правило 7. `non_functioning_hour_has_zero_rentals`

- Поля: связь `Functioning Day` и `Rented Bike Count`.
- Проверяемые строки: часы с `Functioning Day = No` и корректным числом аренд.
- Условие: в час, когда прокат не работал, число аренд равно нулю.
- Порог: число нарушений не больше `max_rentals_when_stopped = 0`.
- Критичность: `critical`. Действие: `quarantine`.
- Связь с задачей: нерабочие часы исключаются из среднего спроса. Аренды в «нерабочий» час означают
  противоречивую запись, которую нельзя отнести ни к рабочим, ни к нерабочим часам.

Для справки ниже посчитаны и рабочие часы без единой аренды: это не нарушение правила, но такой час
заслуживал бы проверки.

In [12]:
max_rentals_when_stopped = 0
stopped_population = """"Functioning Day" = 'No' AND parsed_count IS NOT NULL"""
stopped_violation = "parsed_count <> 0"
checked, violations = connection.execute(f"""
    SELECT count(*), count(*) FILTER (WHERE {stopped_violation})
    FROM parsed_raw
    WHERE {stopped_population}
""").fetchone()
record_rule("non_functioning_hour_has_zero_rentals", stopped_population, stopped_violation,
            checked, violations, max_rentals_when_stopped, "critical", "quarantine",
            "Аренды в час, когда прокат не работал")
show_table(["Рабочих часов", "Из них без аренд", "Минимум аренд в рабочий час"], [
    connection.execute("""
        SELECT count(*), count(*) FILTER (WHERE parsed_count = 0), min(parsed_count)
        FROM parsed_raw WHERE "Functioning Day" = 'Yes'
    """).fetchone()
])

Правило,Проверено,Нарушений,Доля,Порог,Сравнение,Критичность,Действие
non_functioning_hour_has_zero_rentals,295,0,0.00%,count ≤ 0,Не превышен,critical,quarantine


Рабочих часов,Из них без аренд,Минимум аренд в рабочий час
8465,0,2


### Правило 8. `season_matches_month`

- Поля: связь `Seasons` и месяца `Date`.
- Проверяемые строки: строки с корректной датой и сезоном из справочника.
- Условие: сезон совпадает с метеорологическим сезоном месяца: декабрь-февраль - `Winter`, март-май -
  `Spring`, июнь-август - `Summer`, сентябрь-ноябрь - `Autumn`.
- Порог: число нарушений не больше `max_season_mismatch = 0`.
- Критичность: `warning`. Действие: `report_only`.
- Связь с задачей: сезон - срез основного вопроса. Поле источника используется без изменения; если бы
  нарушения нашлись, сравнение сезонов пришлось бы перепроверить по месяцам, но сами часы остаются
  пригодными для остальных срезов.

In [13]:
max_season_mismatch = 0
season_population = """
    parsed_date IS NOT NULL AND "Seasons" IN ('Winter', 'Spring', 'Summer', 'Autumn')
"""
season_violation = """
    "Seasons" <> CASE
        WHEN month(parsed_date) IN (12, 1, 2) THEN 'Winter'
        WHEN month(parsed_date) IN (3, 4, 5) THEN 'Spring'
        WHEN month(parsed_date) IN (6, 7, 8) THEN 'Summer'
        ELSE 'Autumn'
    END
"""
checked, violations = connection.execute(f"""
    SELECT count(*), count(*) FILTER (WHERE {season_violation})
    FROM parsed_raw
    WHERE {season_population}
""").fetchone()
record_rule("season_matches_month", season_population, season_violation, checked, violations,
            max_season_mismatch, "warning", "report_only", "Сезон не совпадает с месяцем")

Правило,Проверено,Нарушений,Доля,Порог,Сравнение,Критичность,Действие
season_matches_month,8760,0,0.00%,count ≤ 0,Не превышен,warning,report_only


### Правило 9. `humidity_in_physical_range`

- Поле: `Humidity(%)`.
- Проверяемые строки: строки с числовым значением влажности.
- Условие: относительная влажность от 1 до 100 %. Нулевая влажность у поверхности земли в Сеуле физически
  неправдоподобна и указывает на сбой метеонаблюдения в этот час.
- Порог: число нарушений не больше `max_bad_humidity = 0`.
- Критичность: `warning`. Действие: `flag`.
- Связь с задачей: на основной вопрос влажность не влияет, поэтому час сохраняется для метрик спроса.
  Погодные признаки такого часа ненадёжны для срезов по осадкам и температуре в уточняющих вопросах:
  флаг позволяет исключить или отдельно рассмотреть эти часы.

In [14]:
max_bad_humidity = 0
humidity_population = "parsed_humidity IS NOT NULL"
humidity_violation = "parsed_humidity < 1 OR parsed_humidity > 100"
checked, violations = connection.execute(f"""
    SELECT count(*), count(*) FILTER (WHERE {humidity_violation})
    FROM parsed_raw
    WHERE {humidity_population}
""").fetchone()
record_rule("humidity_in_physical_range", humidity_population, humidity_violation, checked,
            violations, max_bad_humidity, "warning", "flag", "Влажность вне диапазона 1-100 %")
show_table(["Дата", "Часов с нарушением", "Часы"], connection.execute(f"""
    SELECT parsed_date, count(*), string_agg(parsed_hour::VARCHAR, ', ' ORDER BY parsed_hour)
    FROM parsed_raw
    WHERE {humidity_population} AND ({humidity_violation})
    GROUP BY parsed_date
    ORDER BY parsed_date
""").fetchall())

Правило,Проверено,Нарушений,Доля,Порог,Сравнение,Критичность,Действие
humidity_in_physical_range,8760,17,0.19%,count ≤ 0,Превышен,warning,flag


Дата,Часов с нарушением,Часы
2018-05-19,1,7
2018-05-21,6,"2, 3, 4, 5, 6, 7"
2018-05-22,4,"3, 4, 5, 6"
2018-05-27,2,"5, 7"
2018-05-28,4,"4, 5, 6, 7"


### Сводка правил

Итоговая таблица для отчёта. Все критические правила с действием `quarantine` выполняются без нарушений,
поэтому карантин будет пустым. Порог превышен только у правила `humidity_in_physical_range`
с действием `flag`.

In [15]:
show_table(RULE_HEADERS, [rule["row"] for rule in rules.values()])

Правило,Проверено,Нарушений,Доля,Порог,Сравнение,Критичность,Действие
required_fields_present,8760,0,0.00%,count ≤ 0,Не превышен,critical,quarantine
date_valid_in_period,8760,0,0.00%,count ≤ 0,Не превышен,critical,quarantine
hour_in_range,8760,0,0.00%,count ≤ 0,Не превышен,critical,quarantine
rented_count_non_negative_integer,8760,0,0.00%,count ≤ 0,Не превышен,critical,quarantine
categories_valid,8760,0,0.00%,count ≤ 0,Не превышен,critical,quarantine
date_hour_unique,8760,0,0.00%,count ≤ 0,Не превышен,critical,quarantine
non_functioning_hour_has_zero_rentals,295,0,0.00%,count ≤ 0,Не превышен,critical,quarantine
season_matches_month,8760,0,0.00%,count ≤ 0,Не превышен,warning,report_only
humidity_in_physical_range,8760,17,0.19%,count ≤ 0,Превышен,warning,flag


<a id="prepare"></a>
## Шаг 4. Staging и карантин

Строка попадает в карантин, если нарушает хотя бы одно правило с действием `quarantine`.
В `quality_rule_ids` перечисляются идентификаторы всех таких правил через `;`, в `quarantine_reason` -
их пояснения через `; `. Каждая строка попадает в карантин один раз, карантин имеет приоритет перед флагом.
Карантин сохраняет все 14 исходных полей в исходном строковом виде и с исходными именами.

Остальные строки попадают в staging с явной нормализацией:

- имена приведены к `snake_case` без пробелов, скобок и символа `°`; единица измерения указана суффиксом;
- `Date` разобрана по формату `%d/%m/%Y` в `DATE`, целые поля приведены к `INTEGER`, дробные - к `DOUBLE`;
- `Holiday` и `Functioning Day` заменены логическими признаками `is_holiday` и `is_functioning_hour`
  по однозначному соответствию `Holiday`/`No Holiday` и `Yes`/`No`;
- добавлен признак `quality_humidity_in_physical_range_violated` правила `humidity_in_physical_range`:
  `true` - влажность вне диапазона 1-100 %, `false` - нарушения нет. Значение влажности не меняется.

Агрегаций, отбора нерабочих часов и «исправлений по догадке» нет. Строки упорядочены по дате и часу,
чтобы повторный запуск давал тот же файл.

Сначала проверим, что все погодные поля, которые не проверялись правилами, приводятся к своим типам
однозначно: иначе приведение типа исказило бы или потеряло значение.

In [16]:
cast_failures = connection.execute("""
    SELECT
        count(*) FILTER (WHERE "Temperature(°C)" IS NOT NULL
                         AND TRY_CAST(trim("Temperature(°C)") AS DOUBLE) IS NULL),
        count(*) FILTER (WHERE "Humidity(%)" IS NOT NULL
                         AND NOT regexp_full_match(trim("Humidity(%)"), '[0-9]+')),
        count(*) FILTER (WHERE "Wind speed (m/s)" IS NOT NULL
                         AND TRY_CAST(trim("Wind speed (m/s)") AS DOUBLE) IS NULL),
        count(*) FILTER (WHERE "Visibility (10m)" IS NOT NULL
                         AND NOT regexp_full_match(trim("Visibility (10m)"), '[0-9]+')),
        count(*) FILTER (WHERE "Dew point temperature(°C)" IS NOT NULL
                         AND TRY_CAST(trim("Dew point temperature(°C)") AS DOUBLE) IS NULL),
        count(*) FILTER (WHERE "Solar Radiation (MJ/m2)" IS NOT NULL
                         AND TRY_CAST(trim("Solar Radiation (MJ/m2)") AS DOUBLE) IS NULL),
        count(*) FILTER (WHERE "Rainfall(mm)" IS NOT NULL
                         AND TRY_CAST(trim("Rainfall(mm)") AS DOUBLE) IS NULL),
        count(*) FILTER (WHERE "Snowfall (cm)" IS NOT NULL
                         AND TRY_CAST(trim("Snowfall (cm)") AS DOUBLE) IS NULL)
    FROM raw_data
""").fetchone()
cast_fields = (
    "Temperature(°C)", "Humidity(%)", "Wind speed (m/s)", "Visibility (10m)",
    "Dew point temperature(°C)", "Solar Radiation (MJ/m2)", "Rainfall(mm)", "Snowfall (cm)",
)
show_table(["Поле", "Значений, которые нельзя привести однозначно"],
           zip(cast_fields, cast_failures))
assert sum(cast_failures) == 0, "Есть погодные значения, которые нельзя привести к типу однозначно."

Поле,"Значений, которые нельзя привести однозначно"
Temperature(°C),0
Humidity(%),0
Wind speed (m/s),0
Visibility (10m),0
Dew point temperature(°C),0
Solar Radiation (MJ/m2),0
Rainfall(mm),0
Snowfall (cm),0


In [17]:
critical_ids = [rule_id for rule_id, rule in rules.items() if rule["action"] == "quarantine"]
violation_columns = ",\n        ".join(
    f"COALESCE(({rule['population']}) AND ({rule['violation']}), false) "
    f"AS {identifier('violated_' + rule_id)}"
    for rule_id, rule in rules.items()
)
connection.execute(f"""
    CREATE OR REPLACE TEMP VIEW checked_data AS
    SELECT *,
        {violation_columns}
    FROM parsed_raw
""")
quarantine_condition = " OR ".join(identifier("violated_" + rule_id) for rule_id in critical_ids)

connection.execute(f"""
    CREATE OR REPLACE TEMP VIEW staging_data AS
    SELECT
        parsed_date AS rental_date,
        parsed_hour AS rental_hour,
        parsed_count AS rented_bike_count,
        CAST(trim("Temperature(°C)") AS DOUBLE) AS temperature_c,
        CAST(trim("Humidity(%)") AS INTEGER) AS humidity_pct,
        CAST(trim("Wind speed (m/s)") AS DOUBLE) AS wind_speed_ms,
        CAST(trim("Visibility (10m)") AS INTEGER) AS visibility_10m,
        CAST(trim("Dew point temperature(°C)") AS DOUBLE) AS dew_point_temperature_c,
        CAST(trim("Solar Radiation (MJ/m2)") AS DOUBLE) AS solar_radiation_mj_m2,
        CAST(trim("Rainfall(mm)") AS DOUBLE) AS rainfall_mm,
        CAST(trim("Snowfall (cm)") AS DOUBLE) AS snowfall_cm,
        "Seasons" AS season,
        "Holiday" = 'Holiday' AS is_holiday,
        "Functioning Day" = 'Yes' AS is_functioning_hour,
        "violated_humidity_in_physical_range" AS quality_humidity_in_physical_range_violated
    FROM checked_data
    WHERE NOT ({quarantine_condition})
    ORDER BY rental_date, rental_hour
""")

rule_ids_sql = ", ".join(
    f"CASE WHEN {identifier('violated_' + rule_id)} THEN {literal(rule_id)} END"
    for rule_id in critical_ids
)
reasons_sql = ", ".join(
    f"CASE WHEN {identifier('violated_' + rule_id)} THEN {literal(rules[rule_id]['reason'])} END"
    for rule_id in critical_ids
)
source_columns_sql = ", ".join(identifier(name) for name in raw_columns)
connection.execute(f"""
    CREATE OR REPLACE TEMP VIEW quarantine_data AS
    SELECT {source_columns_sql},
        concat_ws(';', {rule_ids_sql}) AS quality_rule_ids,
        concat_ws('; ', {reasons_sql}) AS quarantine_reason
    FROM checked_data
    WHERE {quarantine_condition}
    ORDER BY parsed_date NULLS LAST, parsed_hour NULLS LAST, "Date", "Hour"
""")

prepared_counts = connection.execute("""
    SELECT
        (SELECT count(*) FROM raw_data),
        (SELECT count(*) FROM staging_data),
        (SELECT count(*) FROM quarantine_data),
        (SELECT count(*) FROM staging_data WHERE quality_humidity_in_physical_range_violated)
""").fetchone()
show_table(["Raw", "Staging", "Карантин", "Флагов в staging"], [prepared_counts])
assert prepared_counts[0] == prepared_counts[1] + prepared_counts[2], "Не сошёлся баланс строк."

Raw,Staging,Карантин,Флагов в staging
8760,8760,0,17


### Примеры записей

Помеченные строки показаны вместе с влажностью, рассчитанной по температуре и точке росы
(приближение Магнуса: `RH = 100 · exp(17.625·Td / (243.04 + Td) − 17.625·T / (243.04 + T))`).
Расчёт приведён только как довод в пользу решения: точка росы не согласуется с нулевой влажностью,
значит, ошибочно именно значение `Humidity(%)`. Поле `humidity_pct` при этом не заменяется.

In [18]:
show_table(
    ["Дата", "Час", "Влажность, %", "Температура, °C", "Точка росы, °C",
     "Влажность по точке росы, %", "Флаг"],
    connection.execute("""
        SELECT rental_date, rental_hour, humidity_pct, temperature_c, dew_point_temperature_c,
               round(100 * exp(
                   17.625 * dew_point_temperature_c / (243.04 + dew_point_temperature_c)
                   - 17.625 * temperature_c / (243.04 + temperature_c)
               ), 1),
               quality_humidity_in_physical_range_violated
        FROM staging_data
        WHERE quality_humidity_in_physical_range_violated
        ORDER BY rental_date, rental_hour
        LIMIT 5
    """).fetchall(),
)

quarantine_rows = connection.execute("""
    SELECT "Date", "Hour", "Rented Bike Count", "Functioning Day",
           quality_rule_ids, quarantine_reason
    FROM quarantine_data
    LIMIT 2
""").fetchall()
if quarantine_rows:
    show_table(["Date", "Hour", "Rented Bike Count", "Functioning Day", "Правила", "Причина"],
               quarantine_rows)
else:
    show_table(["Карантин"],
               [("Строк с критическими нарушениями нет: файл карантина будет пустым",)])

Дата,Час,"Влажность, %","Температура, °C","Точка росы, °C","Влажность по точке росы, %",Флаг
2018-05-19,7,0,11.4,4.5,62.5,True
2018-05-21,2,0,13.9,-2.1,33.1,True
2018-05-21,3,0,13.0,-2.5,34.0,True
2018-05-21,4,0,12.4,-3.4,33.1,True
2018-05-21,5,0,11.9,-2.7,36.0,True


Карантин
Строк с критическими нарушениями нет: файл карантина будет пустым


### Запись двух результатов в S3

Используем способ демонстрационного примера: удаляем только целевые объекты `staging.parquet`
и `quarantine.parquet` и записываем их заново. Повторный запуск заменяет результаты, а не добавляет
строки. Пустой карантин тоже записывается: Parquet-файл сохраняет схему без строк.

In [19]:
staging_bucket, output_key_prefix = OUTPUT_PREFIX.removeprefix("s3://").split("/", 1)
client.head_bucket(Bucket=staging_bucket)
for name in ("staging", "quarantine"):
    key = f"{output_key_prefix}/{name}.parquet"
    client.delete_object(Bucket=staging_bucket, Key=key)
    connection.execute(
        f"COPY (SELECT * FROM {name}_data) TO {literal(f's3://{staging_bucket}/{key}')} "
        "(FORMAT PARQUET)"
    )
show_table(["Результат", "S3-путь", "Размер, байт"], [
    (name, f"{OUTPUT_PREFIX}/{name}.parquet",
     client.head_object(Bucket=staging_bucket,
                        Key=f"{output_key_prefix}/{name}.parquet")["ContentLength"])
    for name in ("staging", "quarantine")
])

Результат,S3-путь,"Размер, байт"
staging,s3://staging/seoul_bike_sharing/source_ingested_on=2026-10-24/staging.parquet,160343
quarantine,s3://staging/seoul_bike_sharing/source_ingested_on=2026-10-24/quarantine.parquet,448


<a id="report"></a>
## Шаг 5. Краткие пояснения для отчёта

Результаты перенесены в [docs/data_quality_report.md](../docs/data_quality_report.md): контекст, профиль,
таблица правил, подготовленные данные, воспроизводимость и ограничения. Описания правил и SQL в отчёте
не повторяются - отчёт ссылается на этот ноутбук.

- Критических нарушений нет: карантин пуст, все 8 760 часов перешли в staging.
- 17 часов с нулевой влажностью сохранены с признаком `quality_humidity_in_physical_range_violated`.
- 295 нерабочих часов сохранены в staging с `is_functioning_hour = false`: это не нарушение качества,
  а состояние сервиса, которое учитывается при расчёте метрик.

<a id="verify"></a>
## Шаг 6. Проверка сохранённых файлов

Читаем оба файла непосредственно из S3 и проверяем на них:

- баланс `raw = staging + карантин` и схемы обоих файлов;
- отсутствие нарушений всех критических правил на staging - уже по нормализованным именам и типам;
- совпадение числа флагов с числом строк, где влажность вне диапазона;
- заполненность причин в карантине;
- неизменность исходного raw-объекта (ETag, размер и время изменения до и после запуска).

In [20]:
for name in ("staging", "quarantine"):
    connection.execute(f"""
        CREATE OR REPLACE TEMP VIEW saved_{name} AS
        SELECT * FROM read_parquet({literal(f'{OUTPUT_PREFIX}/{name}.parquet')},
                                   hive_partitioning = false)
    """)
staging_count = connection.execute("SELECT count(*) FROM saved_staging").fetchone()[0]
quarantine_count = connection.execute("SELECT count(*) FROM saved_quarantine").fetchone()[0]

staging_schema = [row[:2] for row in connection.execute("DESCRIBE saved_staging").fetchall()]
expected_staging_schema = [
    ("rental_date", "DATE"), ("rental_hour", "INTEGER"), ("rented_bike_count", "INTEGER"),
    ("temperature_c", "DOUBLE"), ("humidity_pct", "INTEGER"), ("wind_speed_ms", "DOUBLE"),
    ("visibility_10m", "INTEGER"), ("dew_point_temperature_c", "DOUBLE"),
    ("solar_radiation_mj_m2", "DOUBLE"), ("rainfall_mm", "DOUBLE"), ("snowfall_cm", "DOUBLE"),
    ("season", "VARCHAR"), ("is_holiday", "BOOLEAN"), ("is_functioning_hour", "BOOLEAN"),
    ("quality_humidity_in_physical_range_violated", "BOOLEAN"),
]
quarantine_schema = [row[:2] for row in connection.execute("DESCRIBE saved_quarantine").fetchall()]
expected_quarantine_schema = [(name, "VARCHAR") for name in raw_columns] + [
    ("quality_rule_ids", "VARCHAR"), ("quarantine_reason", "VARCHAR"),
]

critical_on_staging = connection.execute(f"""
    SELECT
        count(*) FILTER (
            WHERE rental_date IS NULL OR rental_hour IS NULL OR rented_bike_count IS NULL
               OR season IS NULL OR is_holiday IS NULL OR is_functioning_hour IS NULL
        ),
        count(*) FILTER (
            WHERE rental_date < DATE {literal(PERIOD_START)}
               OR rental_date >= DATE {literal(PERIOD_END)}
        ),
        count(*) FILTER (WHERE rental_hour NOT BETWEEN 0 AND 23),
        count(*) FILTER (WHERE rented_bike_count < 0),
        count(*) FILTER (WHERE season NOT IN ('Winter', 'Spring', 'Summer', 'Autumn')),
        count(*) - (SELECT count(*) FROM (SELECT DISTINCT rental_date, rental_hour
                                           FROM saved_staging)),
        count(*) FILTER (WHERE NOT is_functioning_hour AND rented_bike_count <> 0)
    FROM saved_staging
""").fetchone()
flag_profile = connection.execute("""
    SELECT
        count(*) FILTER (WHERE quality_humidity_in_physical_range_violated),
        count(*) FILTER (WHERE humidity_pct < 1 OR humidity_pct > 100),
        count(*) FILTER (
            WHERE quality_humidity_in_physical_range_violated
                  IS DISTINCT FROM COALESCE(humidity_pct < 1 OR humidity_pct > 100, false)
        )
    FROM saved_staging
""").fetchone()
empty_reasons = connection.execute("""
    SELECT count(*) FROM saved_quarantine
    WHERE NULLIF(quality_rule_ids, '') IS NULL OR NULLIF(quarantine_reason, '') IS NULL
""").fetchone()[0]
raw_after = client.head_object(Bucket=raw_bucket, Key=raw_key)
raw_unchanged = all(raw_after[field] == raw_before[field]
                    for field in ("ETag", "ContentLength", "LastModified"))

assert raw_count == staging_count + quarantine_count, "Не сошёлся баланс строк."
assert staging_schema == expected_staging_schema, "Схема staging отличается от ожидаемой."
assert quarantine_schema == expected_quarantine_schema, "Схема карантина отличается от ожидаемой."
assert sum(critical_on_staging) == 0, "В staging остались критические нарушения."
assert flag_profile[0] == flag_profile[1] and flag_profile[2] == 0, "Флаги влажности неверны."
assert empty_reasons == 0, "В карантине есть строки без причины."
assert raw_unchanged, "Raw-объект изменился."

show_table(["Критическое правило", "Нарушений в сохранённом staging"],
           zip(critical_ids, critical_on_staging))
show_table(["Флаг TRUE", "Влажность вне 1-100 %", "Несовпадений флага"], [flag_profile])
show_table(["Файл", "Полей", "Схема совпадает с ожидаемой"], [
    ("staging.parquet", len(staging_schema), staging_schema == expected_staging_schema),
    ("quarantine.parquet", len(quarantine_schema),
     quarantine_schema == expected_quarantine_schema),
])
show_table(["Raw-объект", "ETag", "Размер, байт", "Не изменён"],
           [(RAW_PATH, raw_after["ETag"], raw_after["ContentLength"], raw_unchanged)])

run_summary = (
    raw_count, staging_count, quarantine_count, flag_profile[0], sum(critical_on_staging),
)
show_table(["Raw", "Staging", "Карантин", "Флагов в staging", "Критических нарушений в staging"],
           [run_summary])

Критическое правило,Нарушений в сохранённом staging
required_fields_present,0
date_valid_in_period,0
hour_in_range,0
rented_count_non_negative_integer,0
categories_valid,0
date_hour_unique,0
non_functioning_hour_has_zero_rentals,0


Флаг TRUE,Влажность вне 1-100 %,Несовпадений флага
17,17,0


Файл,Полей,Схема совпадает с ожидаемой
staging.parquet,15,True
quarantine.parquet,16,True


Raw-объект,ETag,"Размер, байт",Не изменён
s3://raw/seoul_bike_sharing/ingested_on=2026-10-24/SeoulBikeData.csv,"""bf632307dcb419f67df589faaa0318e1""",604166,True


Raw,Staging,Карантин,Флагов в staging,Критических нарушений в staging
8760,8760,0,17,0


In [21]:
show_table(["Поле staging", "Тип"], staging_schema)
show_table(["Дата", "Час", "Аренд", "Влажность, %", "Сезон", "Праздник", "Прокат работал", "Флаг"],
           connection.execute("""
    SELECT rental_date, rental_hour, rented_bike_count, humidity_pct, season, is_holiday,
           is_functioning_hour, quality_humidity_in_physical_range_violated
    FROM saved_staging
    WHERE quality_humidity_in_physical_range_violated
    ORDER BY rental_date, rental_hour
    LIMIT 2
""").fetchall())
connection.close()

Поле staging,Тип
rental_date,DATE
rental_hour,INTEGER
rented_bike_count,INTEGER
temperature_c,DOUBLE
humidity_pct,INTEGER
wind_speed_ms,DOUBLE
visibility_10m,INTEGER
dew_point_temperature_c,DOUBLE
solar_radiation_mj_m2,DOUBLE
rainfall_mm,DOUBLE


Дата,Час,Аренд,"Влажность, %",Сезон,Праздник,Прокат работал,Флаг
2018-05-19,7,436,0,Spring,False,True,True
2018-05-21,2,262,0,Spring,False,True,True


### Повторное выполнение с чистого ядра

Ноутбук выполнен дважды: после перезапуска ядра все ячейки выполнялись сверху вниз с теми же входным
путём, периодом, порогами и выходным префиксом. Количества обоих выполнений совпали и приведены в отчёте.

Контрольные количества для этого снимка: raw - 8 760, staging - 8 760, карантин - 0, флагов
влажности - 17, критических нарушений в staging - 0.